In [ ]:
import ee
import geopandas as gpd
# from shapely.geometry import mapping
from os.path import exists
from unidecode import unidecode
import json
import concurrent.futures
import pandas as pd

# Trigger the authentication flow.
ee.Authenticate(quiet=True)
# Initialize the library.
ee.Initialize(project='ee-daytime-images')

In [11]:
df = gpd.read_file('/Volumes/ssd/Doutorado/DATASET/IBGE/Favelas e Comunidades Urbanas - 2022/poligonos/fcu_blank/qg_2022_670_fcu_agreg.shp')
geojson = json.loads(df.to_json())
df


,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,geometry
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,"POLYGON ((-46.37112 -23.5, -46.37141 -23.5001,..."
1,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,"POLYGON ((-49.05007 -26.93529, -49.05009 -26.9..."
2,33045570554,Tiqui,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,"POLYGON ((-43.48263 -22.8887, -43.48252 -22.88..."
3,33045570104,Parque Nossa Senhora da Penha,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,"POLYGON ((-43.21664 -22.87497, -43.21662 -22.8..."
4,35095020135,Núcleo Residencial Jardim das Andorinhas II,35,São Paulo,SP,3509502,Campinas,"POLYGON ((-47.01387 -22.91975, -47.01458 -22.9..."
...,...,...,...,...,...,...,...,...
12343,27043020123,Vila Santa Cruz,27,Alagoas,AL,2704302,Maceió,"POLYGON ((-35.76558 -9.57082, -35.76767 -9.572..."
12344,28003080047,Invasão da Jabotiana,28,Sergipe,SE,2800308,Aracaju,"POLYGON ((-37.08826 -10.94031, -37.08824 -10.9..."
12345,35188000223,Residencial Bambi II,35,São Paulo,SP,3518800,Guarulhos,"POLYGON ((-46.39452 -23.37916, -46.3949 -23.37..."
12346,26116060881,Sítio Wanderley,26,Pernambuco,PE,2611606,Recife,"POLYGON ((-34.95091 -8.04458, -34.95088 -8.044..."


In [3]:
so2_collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_SO2')
    .select('SO2_column_number_density')
    .filterDate('2024-01-01', '2024-11-01'))
so2_image = so2_collection.mean()

co_collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CO')
    .select('CO_column_number_density')
    .filterDate('2024-01-01', '2024-11-01'))
co_image = co_collection.mean()

o3_collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_O3')
    .select('O3_column_number_density')
    .filterDate('2024-01-01', '2024-11-01'))
o3_image = o3_collection.mean()

no2_collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
    .select('NO2_column_number_density')
    .filterDate('2024-01-01', '2024-11-01'))
no2_image = no2_collection.mean()

In [4]:
def get_mean(geometry, image, band_name):
    mean_value = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=30,  # Ajuste a escala conforme necessário
        bestEffort=True
    ).get(band_name)
    return mean_value.getInfo()


In [5]:
def process_row(i, row):
    print(i)
    """Processa uma única linha do DataFrame."""
    try:
        geometry = ee.Geometry(row.geometry.__geo_interface__)  # Convertendo a geometria
        so2_mean = get_mean(geometry, so2_image, 'SO2_column_number_density')
        co_mean = get_mean(geometry, co_image, 'CO_column_number_density')
        o3_mean = get_mean(geometry, o3_image, 'O3_column_number_density')
        no2_mean = get_mean(geometry, no2_image, 'NO2_column_number_density')

        # return i, so2_mean, co_mean, o3_mean, no2_mean
        df_result = pd.DataFrame([{
            "id": i,
            "SO2_mean": so2_mean,
            "CO_mean": co_mean,
            "O3_mean": o3_mean,
            "NO2_mean": no2_mean
        }])

        # Salvando CSV individualmente
        file_path = f"./dataset/air_pollution/results/result_{i}.csv"
        df_result.to_csv(file_path, index=False)
    except Exception as e:
        print(f"Erro na linha {i}: {e}")
        return i, None, None, None, None

In [ ]:
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    results = list(executor.map(lambda args: process_row(*args), enumerate(df.itertuples(index=False))))

# print(results)
# for i, so2_mean, co_mean, o3_mean, no2_mean in results:
#     df.loc[i, 'SO2_mean'] = so2_mean
#     df.loc[i, 'CO_mean'] = co_mean
#     df.loc[i, 'O3_mean'] = o3_mean
#     df.loc[i, 'NO2_mean'] = no2_mean

# # Salvar o DataFrame atualizado
# df.to_file('./dataset/air_pollution/df_updated.shp')



In [ ]:
# merge with df files 
for i in range(len(df)):
    file_path = f"./dataset/air_pollution/results/result_{i}.csv"
    if exists(file_path):
        df_result = pd.read_csv(file_path)
        df.loc[i, 'SO2_mean'] = df_result['SO2_mean'].values[0]
        df.loc[i, 'CO_mean'] = df_result['CO_mean'].values[0]
        df.loc[i, 'O3_mean'] = df_result['O3_mean'].values[0]
        df.loc[i, 'NO2_mean'] = df_result['NO2_mean'].values[0]
        # break

# df.to_file('./dataset/air_pollution/df_updated.shp')
df

NameError: name 'results' is not defined

In [15]:
df.to_file('./dataset/air_pollution/air_pollution.shp')

# OTHERS

In [ ]:
# for in df
for i, row in df.iterrows():
    print(i)
    geometry = ee.Geometry(geojson["features"][i]["geometry"])
    so2_mean = get_mean(geometry, so2_image, 'SO2_column_number_density')
    co_mean = get_mean(geometry, co_image, 'CO_column_number_density')
    o3_mean = get_mean(geometry, o3_image, 'O3_column_number_density')
    no2_mean = get_mean(geometry, no2_image, 'NO2_column_number_density')

    print('calculou')

    df.loc[i, 'SO2_mean'] = so2_mean
    df.loc[i, 'CO_mean'] = co_mean
    df.loc[i, 'O3_mean'] = o3_mean
    df.loc[i, 'NO2_mean'] = no2_mean

    df.to_file('./dataset/air_pollution/df.shp')  

In [129]:
df

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,geometry,SO2_mean,CO_mean,O3_mean,NO2_mean
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,"POLYGON ((-46.37112 -23.5, -46.37141 -23.5001,...",0.000226,0.032436,0.117354,0.000139
1,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,"POLYGON ((-49.05007 -26.93529, -49.05009 -26.9...",0.000128,0.030196,0.119900,0.000063
2,33045570554,Tiqui,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,"POLYGON ((-43.48263 -22.8887, -43.48252 -22.88...",0.000190,0.034092,0.117908,0.000099
3,33045570104,Parque Nossa Senhora da Penha,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,"POLYGON ((-43.21664 -22.87497, -43.21662 -22.8...",0.000173,0.035577,0.118095,0.000126
4,35095020135,Núcleo Residencial Jardim das Andorinhas II,35,São Paulo,SP,3509502,Campinas,"POLYGON ((-47.01387 -22.91975, -47.01458 -22.9...",0.000167,0.032145,0.117010,0.000090
...,...,...,...,...,...,...,...,...,...,...,...,...
12343,27043020123,Vila Santa Cruz,27,Alagoas,AL,2704302,Maceió,"POLYGON ((-35.76558 -9.57082, -35.76767 -9.572...",NaN,NaN,NaN,NaN
12344,28003080047,Invasão da Jabotiana,28,Sergipe,SE,2800308,Aracaju,"POLYGON ((-37.08826 -10.94031, -37.08824 -10.9...",NaN,NaN,NaN,NaN
12345,35188000223,Residencial Bambi II,35,São Paulo,SP,3518800,Guarulhos,"POLYGON ((-46.39452 -23.37916, -46.3949 -23.37...",NaN,NaN,NaN,NaN
12346,26116060881,Sítio Wanderley,26,Pernambuco,PE,2611606,Recife,"POLYGON ((-34.95091 -8.04458, -34.95088 -8.044...",NaN,NaN,NaN,NaN


In [ ]:
def get_so2(name, geometry):
    collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_SO2')
                  .select('SO2_column_number_density')
                  .filterDate('2024-01-01', '2024-11-01')
                  .filterBounds(geometry))
    
    image = collection.mean()
    image_name = name.replace(' ', '-')
    
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=image_name,
        folder='SO2',
        scale=30,
        crs='EPSG:4326',
        fileFormat='GeoTIFF',
        region=geometry
    )
    task.start()

In [53]:
def get_no2(city_name, geometry):
    collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
                  .select('NO2_column_number_density')
                  .filterDate('2024-01-01', '2024-11-01')
                  .filterBounds(geometry))
    
    image = collection.mean()
    image_name = 'NO2_' + city_name.replace(' ', '-')
    
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=image_name,
        folder='NO2',
        scale=30,
        crs='EPSG:4326',
        fileFormat='GeoTIFF',
        region=geometry
    )
    task.start()

In [54]:
def get_o3(city_name, geometry):
    collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_O3')
                  .select('O3_column_number_density')
                  .filterDate('2024-01-01', '2024-11-01')
                  .filterBounds(geometry))
    
    image = collection.mean()
    image_name = 'O3_' + city_name.replace(' ', '-')
    
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=image_name,
        folder='O3',
        scale=30,
        crs='EPSG:4326',
        fileFormat='GeoTIFF',
        region=geometry
    )
    task.start()

In [55]:
def get_co(city_name, geometry):
    collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CO')
                  .select('CO_column_number_density')
                  .filterDate('2024-01-01', '2024-11-01')
                  .filterBounds(geometry))
    
    image = collection.mean()
    image_name = 'CO_' + city_name.replace(' ', '-')
    
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=image_name,
        folder='CO',
        scale=30,
        crs='EPSG:4326',
        fileFormat='GeoTIFF',
        region=geometry
    )
    task.start()

KeyboardInterrupt: 